Assignment 3: Retrieval-Augmented Generation (RAG)

In [4]:
!pip install langchain langchain-google-genai langchain-community langchain-core pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 17.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 100.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninsta

In [5]:

from langchain_google_genai import ChatGoogleGenerativeAI
import os
import getpass

In [6]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API Key")

Enter your API Key··········


In [7]:

llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)


### Part 1 — Data Understanding & Preprocessing

In [8]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_450/3076486969.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [11]:
loader=PyPDFLoader("/troubleshooting_runbook.pdf")

In [12]:
documents=loader.load()

report=""

for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))

Total number of pages 7


In [13]:
print(report[:1000])

Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team  |  Version: 3.2  |  Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-service, search-service, reporting-engine and user-profile-api. It covers the ten most frequent
error codes seen in the incident log over the last two quarters. Each error code has a symptom description,
likely causes, a diagnostic checklist, a fix, and a prevention note.
2. How to use this runbook
Start from the error code in the alert or log line. Confirm the symptom matches, work through the
diagnostic checklist in order, and apply the fix only after the cause is confirmed. If two consecutive
diagnostic steps produce no signal,

In [17]:


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks=text_splitter.split_documents(documents)



### Part 2 — Embedding & Vector Database

In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

In [16]:


from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)



In [19]:
# Define the sample query
sample_query = "How do I resolve issues with payment-gateway or database connection?"

# Perform retrieval from our FAISS vector store
retrieved_docs = vector_store.similarity_search(sample_query, k=2)


print("Example of Retrieved Chunks for a Sample Query")
print(f"Sample Query: '{sample_query}'\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content preview:\n{doc.page_content[:400]}...")
    print("\n")

Example of Retrieved Chunks for a Sample Query
Sample Query: 'How do I resolve issues with payment-gateway or database connection?'

--- Chunk 1 ---
Source: /troubleshooting_runbook.pdf
Content preview:
Confirm circuit breaker state and failure threshold. 4. Look for DNS resolution failures in the sidecar logs.
Fix: Fail over to the secondary provider where one exists, enable degraded mode (queue and retry later),
and communicate expected recovery time to stakeholders. Do not repeatedly restart healthy pods.
ERR-504 - Gateway timeout
Symptom: HTTP 504 after 30 seconds; p95 latency above 3,000 ms....


--- Chunk 2 ---
Source: /troubleshooting_runbook.pdf
Content preview:
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 5
Database Errors
DB-1205 - Lock wait timeout exceeded
Symptom: Queries fail with 'Lock wait timeout exceeded; try restarting transaction'. Latency climbs to 2
seconds or more and then writes begin to fail. Seen on payment-

Description of Embedding Model Used:


We are using 'gemini-embedding-2' via langchain-google-genai.
This model converts textual chunks into dense vector representations designed specifically for semantic search and retrieval tasks, capturing semantic meanings effectively.


 Explanation of Vector Storage Approach:


We utilize FAISS (Facebook AI Similarity Search), an efficient library for dense vector similarity search.
FAISS indexes the computed embeddings to allow near-instantaneous similarity calculations (using distance metrics like L2 distance or inner product) when processing user queries.


### Part 3 — Retrieval Pipeline

In [21]:
sample_query = "How do I resolve issues with payment-gateway or database connection?"

In [20]:
results=vector_store.similarity_search_with_score(query=sample_query)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"Score:{score}")
  print(f"Document:{doc.page_content[:500]}")

RESULT0
Score:0.5346267819404602
Document:Confirm circuit breaker state and failure threshold. 4. Look for DNS resolution failures in the sidecar logs.
Fix: Fail over to the secondary provider where one exists, enable degraded mode (queue and retry later),
and communicate expected recovery time to stakeholders. Do not repeatedly restart healthy pods.
ERR-504 - Gateway timeout
Symptom: HTTP 504 after 30 seconds; p95 latency above 3,000 ms. The week of 10 August 2026 saw a
cluster of ERR-504 incidents on payment-gateway, traced to slow qu
RESULT1
Score:0.57561194896698
Document:Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 5
Database Errors
DB-1205 - Lock wait timeout exceeded
Symptom: Queries fail with 'Lock wait timeout exceeded; try restarting transaction'. Latency climbs to 2
seconds or more and then writes begin to fail. Seen on payment-gateway, inventory-service and order-api.
Likely causes: Long-running transactions holding row l

Experiment with different values of k (e.g., 3, 5, 7

In [24]:
retriever_k3=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)


In [25]:
retriever_k5=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [26]:
retriever_k7=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":7}
)

### Part 4 — Answer Generation (RAG)

In [27]:
from langchain_core.prompts import ChatPromptTemplate

In [28]:
rag_prompt = ChatPromptTemplate.from_template(
    """
    You are a Senior Systems Support Engineer. Your job is to answer user questions regarding production incidents and troubleshooting steps based on the retrieved troubleshooting runbook context.

    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context:
    {context}

    Question: {question}

    Provide:
    1. Answer (Step-by-step diagnostic and resolution instructions)
    2. Supporting Evidence (Direct quotes or sections from the runbook context)
    """
)

In [29]:
# Show prompt structure
print(rag_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n    You are a Senior Systems Support Engineer. Your job is to answer user questions regarding production incidents and troubleshooting steps based on the retrieved troubleshooting runbook context.\n\n    If the answer is not available or cannot be found in the context then say - "I couldn\'t find this information"\n\n    Retrieved Context:\n    {context}\n\n    Question: {question}\n\n    Provide:\n    1. Answer (Step-by-step diagnostic and resolution instructions)\n    2. Supporting Evidence (Direct quotes or sections from the runbook context)\n    '), additional_kwargs={})]


In [32]:
from langchain_core.output_parsers import StrOutputParser


In [31]:

rag_chain=rag_prompt|llm|StrOutputParser()

In [33]:
question="Talk about the service map details"

retrieved_docs=retriever_k3.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question,
    }
)

In [34]:
print(response)

Based on the provided troubleshooting runbook context, here are the service map details:

### 1. Answer

The Service Map defines each production service alongside its domain, owning team, primary datastore, and critical dependencies. Use these details to route escalations to the correct team and map out failure chains during incident diagnosis:

*   **payment-gateway**
    *   **Domain:** Payments
    *   **Owning Team:** Team Phoenix
    *   **Primary Datastore:** PostgreSQL (payments-db)
    *   **Critical Dependency:** Bank API, auth-service
*   **auth-service**
    *   **Domain:** Identity
    *   **Owning Team:** Team Sentinel
    *   **Primary Datastore:** Redis + PostgreSQL
    *   **Critical Dependency:** Identity provider
*   **order-api**
    *   **Domain:** Commerce
    *   **Owning Team:** Team Atlas
    *   **Primary Datastore:** PostgreSQL (orders-db)
    *   **Critical Dependency:** inventory-service, payment-gateway
*   **inventory-service**
    *   **Domain:** Commerce

In [40]:
question="Talk about the service map details"

retrieved_docs=retriever_k5.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response_5=rag_chain.invoke({
    "context":context,
    "question":question,
    }
)

In [39]:
print(response_5)

Based on the provided troubleshooting runbook context, here are the details from the **Service Map** section:

### 1. Answer

The Service Map details the services across the platform, including their business domain, owning team, primary datastore, and critical dependencies:

*   **payment-gateway**
    *   **Domain:** Payments
    *   **Owning team:** Team Phoenix
    *   **Primary datastore:** PostgreSQL (payments-db)
    *   **Critical dependency:** Bank API, auth-service
*   **auth-service**
    *   **Domain:** Identity
    *   **Owning team:** Team Sentinel
    *   **Primary datastore:** Redis + PostgreSQL
    *   **Critical dependency:** Identity provider
*   **order-api**
    *   **Domain:** Commerce
    *   **Owning team:** Team Atlas
    *   **Primary datastore:** PostgreSQL (orders-db)
    *   **Critical dependency:** inventory-service, payment-gateway
*   **inventory-service**
    *   **Domain:** Commerce
    *   **Owning team:** Team Atlas
    *   **Primary datastore:** Pos

In [41]:
question="Talk about the service map details"

retrieved_docs=retriever_k7.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response_7=rag_chain.invoke({
    "context":context,
    "question":question,
    }
)

In [42]:
print(response_7)

Here is the summary of the service map details based on the provided runbook:

### 1. Answer

The service map outlines the core services, their respective domains, owning teams, primary datastores, and critical dependencies:

*   **payment-gateway**
    *   **Domain:** Payments
    *   **Owning team:** Team Phoenix
    *   **Primary datastore:** PostgreSQL (payments-db)
    *   **Critical dependency:** Bank API, auth-service
*   **auth-service**
    *   **Domain:** Identity
    *   **Owning team:** Team Sentinel
    *   **Primary datastore:** Redis + PostgreSQL
    *   **Critical dependency:** Identity provider
*   **order-api**
    *   **Domain:** Commerce
    *   **Owning team:** Team Atlas
    *   **Primary datastore:** PostgreSQL (orders-db)
    *   **Critical dependency:** inventory-service, payment-gateway
*   **inventory-service**
    *   **Domain:** Commerce
    *   **Owning team:** Team Atlas
    *   **Primary datastore:** PostgreSQL (inventory-db)
    *   **Critical dependenc